# Workshop 3 Lab — Expert
### Computation 101 · IQ Biology Bootcamp 2026

You're about to reconstruct a real RNA-seq analysis in Google Colab — a full Linux computer in your
browser. **Run each cell in order** (Shift+Enter). Where you see `...`, replace it with your code before running.

*Background:* **ZFP36L2** is an RNA-binding protein that grabs certain messenger RNAs and marks them for
destruction. Knock the gene out in a mouse (**KO**), and those mRNAs are no longer destroyed — so they go
**up**. Researchers measured this in six tissues with RNA-seq, then used DESeq2 to score every gene. A gene
counts as **up-regulated** when its `log2FoldChange > 1` *and* `padj < 0.05`.

**When a cell turns red — read it, don't panic.** Python puts the real problem on the *last* line:
`KeyError: 'padj'` means there's no column by that name; `NameError: name 'up' is not defined` means you
skipped the cell that made `up`. Read bottom-up, fix that one thing, rerun. Still stuck after a couple
tries? **Bring it to the workshop — that's exactly what it's for.**

In [ ]:
# One-time setup. The lines starting with ! are REAL shell commands running in this
# Colab Linux machine — the same commands you'd type on Fiji. This clones the course
# repo (only if it isn't here yet) and points DATA at the ZFP36L2 dataset inside it.
![ -d iqbio-computation-101-2026 ] || git clone --depth 1 https://github.com/gsstephenson/iqbio-computation-101-2026.git
import pandas as pd, os
DATA = 'iqbio-computation-101-2026/data/zfp36l2'
print('data folder contains:', os.listdir(DATA))

## Your job: reproduce a published number, then extend
Minimal scaffolding. Rebuild the six up-sets yourself, then answer: **how many genes are up in four
or more tissues?** Cross-check against the paper's derived table, then pick an extension.

In [ ]:
from collections import Counter
tissues = ['Lung', 'Liver', 'BM', 'Spleen', 'Ovary', 'Kidney']
# 1. build up[t] = set of up-regulated gene IDs for each tissue
# 2. count how many tissues each gene is up in (Counter over all sets)
# 3. how many genes are up in >= 4 tissues?
up = {}
# YOUR CODE HERE


Cross-check against `derived/up_tissue_overlap.csv`, which records each gene's tissue count directly:

In [ ]:
ov = pd.read_csv(os.path.join(DATA, 'derived', 'up_tissue_overlap.csv'))
print((ov['n_tissues_up'] >= 4).sum(), 'genes with n_tissues_up >= 4 (from the table)')
ov.head()

**Extensions** (pick one): (a) which tissues does the #2 gene, *Gm32051*, miss? (b) redo the whole
analysis for **down**-regulated genes (`log2FoldChange < -1`). (c) plot the distribution of
`n_tissues_up`. (d) of the 16 multi-tissue genes, how many are direct eCLIP targets?

In [ ]:
# YOUR CODE HERE  (your extension)


## Plot the shape of the result
You counted genes up in >= 4 tissues. Now show the whole distribution of `n_tissues_up` — how many genes
are up in 1, 2, ... 6 tissues? (A log y-axis helps: most genes are tissue-specific.)

In [ ]:
import matplotlib.pyplot as plt
ov = pd.read_csv(os.path.join(DATA, 'derived', 'up_tissue_overlap.csv'))
counts = ov['n_tissues_up'].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(counts.index, ...)            # the bar heights
plt.xlabel('number of tissues a gene is up in'); plt.ylabel('genes'); plt.yscale('log')
plt.title('Sharing across tissues'); plt.show()

## Your turn — ask your own question
You can reproduce the paper's numbers. Now be the scientist: pick a question the notebook *didn't* answer and
chase it — enrichment on the genes up in >= 4 tissues, up- vs down-regulated enrichment, fold change vs tissue
count, anything. Verify it against something you can check. Here's one worked example to start from:

In [ ]:
# Worked example: are the genes up in >= 4 tissues a coherent program?
!pip install gseapy -q
import gseapy as gp
ov = pd.read_csv(os.path.join(DATA, 'derived', 'up_tissue_overlap.csv'))
shared = ov[ov['n_tissues_up'] >= 4]['gene_id'].tolist()
bm = pd.read_csv(os.path.join(DATA, 'de', 'BM_DE.csv'), index_col=0)   # any table that carries Gene.name
syms = bm.loc[bm.index.isin(shared), 'Gene.name'].dropna().tolist()
enr = gp.enrichr(gene_list=syms, gene_sets=['GO_Biological_Process_2021'], organism='mouse')
print(enr.results.sort_values('Adjusted P-value')[['Term','Adjusted P-value']].head(6).to_string(index=False))

## One last lesson: the statistic won't save you
You just drew pictures of your data. Here's *why* that matters — a trick every bioinformatician meets once.
Four tiny datasets. First, the numbers:

In [ ]:
import numpy as np, matplotlib.pyplot as plt
x123 = [10,8,13,9,11,14,6,4,12,7,5]; x4 = [8,8,8,8,8,8,8,19,8,8,8]
quartet = {'I':(x123,[8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]),
           'II':(x123,[9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]),
           'III':(x123,[7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]),
           'IV':(x4,[6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89])}
for k,(x,y) in quartet.items():
    x=np.array(x,float); y=np.array(y,float); m,b=np.polyfit(x,y,1)
    print(f'{k:3}  mean_x={x.mean():.1f}  mean_y={y.mean():.2f}  corr={np.corrcoef(x,y)[0,1]:.3f}  fit: y={m:.2f}x+{b:.2f}')

Identical — same means, near-identical correlation (≈0.816), the same best-fit line. By the numbers, one dataset.
Now **look**:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7, 6))
for ax,(k,(x,y)) in zip(axes.ravel(), quartet.items()):
    x=np.array(x,float); y=np.array(y,float); m,b=np.polyfit(x,y,1)
    ax.scatter(x, y, c='crimson'); ax.plot([2,20],[m*2+b, m*20+b], 'k--', lw=0.8)
    ax.set_title(f'Set {k}'); ax.set_xlim(2,20); ax.set_ylim(2,14)
plt.tight_layout(); plt.show()

Same statistics, four different truths: **I** a real linear trend; **II** an obvious curve (a straight line
is the *wrong model*); **III** a clean line wrecked by one **outlier**; **IV** a single high-leverage point
that invents the whole correlation. This is **Anscombe's Quartet** (1973) — and it is the entire reason you
plot. The number never warned you; you had to look.

It's also why RNA-seq lives in **log2 space** and why DESeq2 normalizes by median-of-ratios instead of raw
ratios: choose the right transform, then *validate by eye* with a volcano or MA plot — exactly what you just
did. Don't hedge the statistic. Look at your data, pick the right normalization, and trust the picture.

## One more thing

The dataset you just analyzed isn't a textbook toy. It's a real, published study —
*The RNA binding protein ZFP36L2 displays tissue-selective mRNA targeting in mice*, **RNA Biology (2026)**.

Its first author is **George** — a first-year in your own program, who a year ago sat exactly where
you're sitting now. Everything you just reproduced, a peer generated. That's the whole arc of this
bootcamp: the person one year ahead of you did real science, and today you re-derived their headline
result from their deposited data. Next year, someone reproduces yours.